# Train an MLX Object-Detection Model

This notebook runs in a local Jupyter environment or Google Colab. It downloads a ZIP archive containing one YOLO-format dataset, validates the extracted layout, and trains through MLX's object-detection command API.

The archive may contain the dataset at its root or inside one wrapper directory. The dataset must contain `data.yaml`, `images/train`, `images/val`, `labels/train`, and `labels/val`. Run the cells from top to bottom.

## 1. Configuration

Set `DATASET_ZIP_URL` to a directly downloadable HTTP(S) ZIP file. Checkpoints can be kept in the local notebook workspace or under Google Drive when running in Colab. Use a new `EXPERIMENT_NAME` for a fresh run.

In [ ]:
# Dataset download
DATASET_ZIP_URL = ""  # Example: https://example.com/my-yolo-dataset.zip

# MLX source and provider
REPO_URL = "https://github.com/ralampay/mlx.git"
PROVIDER = "ultralytics"  # "ultralytics" or "libreyolo"
MODEL = "yolo26"          # Use "yolo9-t" with the LibreYOLO provider
INSTALL_MISSING_DEPENDENCIES = True

# Training
EPOCHS = 100
BATCH_SIZE = 16
IMAGE_HEIGHT = 640
IMAGE_WIDTH = 640
DEVICE = "auto"           # "auto", "cpu", "cuda", or "cuda:0"
PRETRAINED = False
USE_BEST = True
RANDOM_SEED = 42

# Checkpoints
CHECKPOINT_STORAGE = "local"  # "local" or "google_drive"
RESUME_LATEST = True
EXPERIMENT_NAME = "object-detection"
LOCAL_WORKSPACE_ROOT = ""  # Empty uses <repo>/tmp/notebooks/train_object_detection
GOOGLE_DRIVE_CHECKPOINT_ROOT = "/content/drive/MyDrive/mlx/object_detection/checkpoints"

## 2. Prepare the runtime

A local notebook reuses the surrounding MLX checkout. A standalone Colab notebook clones MLX when necessary. Missing provider dependencies are installed from the forks pinned by this project.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path


def is_mlx_checkout(path: Path) -> bool:
    return (
        (path / "pyproject.toml").is_file()
        and (path / "mlx" / "modes" / "object_detection").is_dir()
    )


def find_mlx_checkout() -> Path | None:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents, current / "mlx"):
        if is_mlx_checkout(candidate):
            return candidate
    return None


try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True


def import_pytorch_runtime():
    try:
        import torch
        import torchvision
    except (AttributeError, ImportError, OSError, RuntimeError) as exc:
        recovery = (
            " In Google Colab, select Runtime > Disconnect and delete runtime, "
            "reconnect, and run the notebook from the first cell. Also remove any "
            "file named torch.py or directory named torch from /content."
            if IN_COLAB
            else " Restart the Jupyter kernel and check for a local torch.py or torch directory."
        )
        raise RuntimeError(
            f"PyTorch and torchvision could not be imported together: {exc}.{recovery}"
        ) from exc

    if not hasattr(torch, "library"):
        raise RuntimeError(
            f"PyTorch loaded from {torch.__file__}, but torch.library is unavailable. "
            "The module is incomplete or shadowed. "
            + (
                "Select Runtime > Disconnect and delete runtime, reconnect, and run "
                "the notebook from the first cell."
                if IN_COLAB
                else "Restart the kernel and check the working directory for torch.py."
            )
        )
    return torch, torchvision

if PROVIDER not in {"ultralytics", "libreyolo"}:
    raise ValueError(
        "PROVIDER must be 'ultralytics' or 'libreyolo'. "
        f"Received: {PROVIDER!r}"
    )

REPO_ROOT = find_mlx_checkout()
if REPO_ROOT is None:
    clone_root = Path("/content") if IN_COLAB else Path.cwd().resolve()
    clone_target = clone_root / "mlx"
    if clone_target.exists():
        raise RuntimeError(
            f"Cannot clone MLX because {clone_target} already exists but is not an MLX checkout. "
            "Move it, or start the notebook from an existing MLX checkout."
        )
    subprocess.check_call(
        ["git", "clone", "--depth", "1", REPO_URL, str(clone_target)]
    )
    REPO_ROOT = clone_target.resolve()

repo_text = str(REPO_ROOT)
if repo_text not in sys.path:
    sys.path.insert(0, repo_text)

# Import Colab's bundled PyTorch before pip resolves provider dependencies.
# This keeps the live kernel on Colab's matched torch/torchvision CUDA build.
if IN_COLAB:
    torch_runtime, torchvision_runtime = import_pytorch_runtime()
else:
    torch_runtime = torchvision_runtime = None

dependency_specs = {
    "ultralytics": (
        "ultralytics",
        "ultralytics @ git+https://github.com/ralampay/ultralytics",
    ),
    "libreyolo": (
        "libreyolo",
        "libreyolo[onnx] @ git+https://github.com/ralampay/libreyolo.git@release",
    ),
}
required_packages = [
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("rich", "rich>=13.7.0"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    dependency_specs[PROVIDER],
]
missing_specs = [
    package_spec
    for module_name, package_spec in required_packages
    if importlib.util.find_spec(module_name) is None
]
if missing_specs:
    if not INSTALL_MISSING_DEPENDENCIES:
        raise RuntimeError(
            "Missing notebook dependencies: "
            + ", ".join(missing_specs)
            + ". Install them or set INSTALL_MISSING_DEPENDENCIES=True."
        )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade-strategy",
            "only-if-needed",
            *missing_specs,
        ]
    )
    importlib.invalidate_caches()

if torch_runtime is None or torchvision_runtime is None:
    torch_runtime, torchvision_runtime = import_pytorch_runtime()

print(f"Runtime: {'Google Colab' if IN_COLAB else 'Jupyter'}")
print(f"MLX checkout: {REPO_ROOT}")
print(f"Provider: {PROVIDER}")
print(
    f"PyTorch: {torch_runtime.__version__} ({torch_runtime.__file__}) | "
    f"torchvision: {torchvision_runtime.__version__}"
)

## 3. Select checkpoint storage

Google Drive mounting is intentionally limited to Colab. In local Jupyter, set `LOCAL_WORKSPACE_ROOT` to any local or already-synced directory. Dataset files stay in the local runtime workspace so training does not repeatedly read images across the Drive mount.

In [ ]:
from mlx.core.exceptions import MLXUserError

storage_mode = CHECKPOINT_STORAGE.strip().lower()
if storage_mode not in {"local", "google_drive"}:
    raise MLXUserError(
        "CHECKPOINT_STORAGE must be 'local' or 'google_drive'. "
        f"Received: {CHECKPOINT_STORAGE!r}"
    )

if not EXPERIMENT_NAME or Path(EXPERIMENT_NAME).name != EXPERIMENT_NAME:
    raise MLXUserError(
        "EXPERIMENT_NAME must be one non-empty directory name without path separators."
    )

if LOCAL_WORKSPACE_ROOT:
    WORKSPACE_ROOT = Path(LOCAL_WORKSPACE_ROOT).expanduser().resolve()
else:
    WORKSPACE_ROOT = REPO_ROOT / "tmp" / "notebooks" / "train_object_detection"
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

if storage_mode == "google_drive":
    if not IN_COLAB:
        raise MLXUserError(
            "Google Drive mounting is only available in Colab. For local Jupyter, "
            "set CHECKPOINT_STORAGE='local' and point LOCAL_WORKSPACE_ROOT at a synced directory."
        )
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
    checkpoint_root = Path(GOOGLE_DRIVE_CHECKPOINT_ROOT).expanduser()
    if not checkpoint_root.is_absolute():
        raise MLXUserError(
            "GOOGLE_DRIVE_CHECKPOINT_ROOT must be an absolute path beneath the mounted Drive."
        )
else:
    checkpoint_root = WORKSPACE_ROOT / "checkpoints"

CHECKPOINT_ROOT = checkpoint_root.resolve()
PROJECT_DIR = CHECKPOINT_ROOT / EXPERIMENT_NAME
RUN_NAME = "train"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Checkpoint project: {PROJECT_DIR}")

## 4. Download, extract, and validate the dataset

The URL is hashed into the local archive and extraction directory names. Rerunning this cell reuses a completed download and extraction, while changing the URL creates a separate cached dataset. ZIP members are checked before extraction to prevent writes outside the dataset workspace.

In [ ]:
import hashlib
import shutil
import zipfile
from collections.abc import Mapping
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

import yaml


class PrepareObjectDetectionDataset:
    def __init__(self, *, archive_url: str, workspace: Path) -> None:
        self.archive_url = archive_url.strip()
        self.workspace = workspace

    def execute(self) -> Path:
        self._validate_url()
        source_id = hashlib.sha256(self.archive_url.encode("utf-8")).hexdigest()[:12]
        archive_path = self.workspace / "downloads" / f"dataset-{source_id}.zip"
        extraction_root = self.workspace / "datasets" / f"dataset-{source_id}"
        self._download(archive_path)
        self._extract(archive_path, extraction_root)
        return self._find_and_validate_dataset(extraction_root)

    def _validate_url(self) -> None:
        if not self.archive_url:
            raise MLXUserError(
                "DATASET_ZIP_URL is empty. Set it to a directly downloadable HTTP(S) ZIP file."
            )
        parsed = urlparse(self.archive_url)
        if parsed.scheme not in {"http", "https"} or not parsed.netloc:
            raise MLXUserError(
                "DATASET_ZIP_URL must be a directly downloadable HTTP(S) URL. "
                f"Received: {self.archive_url!r}"
            )

    def _download(self, archive_path: Path) -> None:
        if archive_path.is_file():
            print(f"Reusing downloaded archive: {archive_path}")
            return
        archive_path.parent.mkdir(parents=True, exist_ok=True)
        partial_path = archive_path.with_suffix(".zip.part")
        request = Request(self.archive_url, headers={"User-Agent": "MLX-notebook"})
        print(f"Downloading dataset from {self.archive_url}")
        try:
            with urlopen(request, timeout=60) as response, partial_path.open("wb") as output:
                shutil.copyfileobj(response, output)
            partial_path.replace(archive_path)
        except (HTTPError, URLError, TimeoutError, OSError) as exc:
            partial_path.unlink(missing_ok=True)
            raise MLXUserError(
                f"Could not download the dataset ZIP from {self.archive_url}: {exc}. "
                "Check that the URL is public and points directly to a ZIP file."
            ) from exc

    def _extract(self, archive_path: Path, extraction_root: Path) -> None:
        if extraction_root.is_dir() and any(extraction_root.rglob("data.yaml")):
            print(f"Reusing extracted dataset: {extraction_root}")
            return
        extraction_root.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(archive_path) as archive:
                root = extraction_root.resolve()
                for member in archive.infolist():
                    target = (root / member.filename).resolve()
                    if target != root and root not in target.parents:
                        raise MLXUserError(
                            f"Unsafe path in dataset ZIP: {member.filename!r}. "
                            "Create an archive without absolute paths or '..' traversal."
                        )
                archive.extractall(root)
        except zipfile.BadZipFile as exc:
            raise MLXUserError(
                f"Downloaded file is not a valid ZIP archive: {archive_path}. "
                "Check that DATASET_ZIP_URL points directly to the archive."
            ) from exc

    def _find_and_validate_dataset(self, extraction_root: Path) -> Path:
        yaml_paths = sorted(
            path
            for path in extraction_root.rglob("data.yaml")
            if "__MACOSX" not in path.parts
        )
        if not yaml_paths:
            raise MLXUserError(
                f"No data.yaml was found under {extraction_root}. "
                "The ZIP must contain one YOLO-format dataset."
            )
        if len(yaml_paths) > 1:
            candidates = ", ".join(str(path.relative_to(extraction_root)) for path in yaml_paths)
            raise MLXUserError(
                "The ZIP contains multiple data.yaml files. Package one dataset per archive. "
                f"Found: {candidates}"
            )

        data_yaml = yaml_paths[0]
        try:
            payload = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
        except (OSError, UnicodeError, yaml.YAMLError) as exc:
            raise MLXUserError(
                f"Could not read YOLO dataset configuration {data_yaml}: {exc}."
            ) from exc
        if not isinstance(payload, Mapping):
            raise MLXUserError(f"Expected a YAML mapping in {data_yaml}.")
        missing_keys = [key for key in ("train", "val", "names") if key not in payload]
        if missing_keys:
            raise MLXUserError(
                f"Dataset YAML {data_yaml} is missing required keys: {', '.join(missing_keys)}."
            )

        declared_root = Path(str(payload.get("path", "."))).expanduser()
        if not declared_root.is_absolute():
            declared_root = (data_yaml.parent / declared_root).resolve()
        extraction_path = extraction_root.resolve()
        if declared_root != extraction_path and extraction_path not in declared_root.parents:
            raise MLXUserError(
                f"The path declared by {data_yaml} resolves outside the extracted archive: "
                f"{declared_root}. Use a portable relative path such as '.'."
            )

        for split in ("train", "val"):
            raw_split = payload[split]
            if not isinstance(raw_split, str) or raw_split.startswith(("http://", "https://")):
                raise MLXUserError(
                    f"Dataset YAML key '{split}' must be one local path inside the ZIP."
                )
            image_path = (declared_root / raw_split).resolve()
            if not image_path.exists():
                raise MLXUserError(
                    f"Dataset split '{split}' does not exist: {image_path}. "
                    "Check data.yaml and the archive layout."
                )
            image_parts = list(image_path.parts)
            if "images" not in image_parts:
                raise MLXUserError(
                    f"Dataset split '{split}' must point beneath an images directory: {image_path}."
                )
            images_index = len(image_parts) - 1 - image_parts[::-1].index("images")
            label_parts = image_parts.copy()
            label_parts[images_index] = "labels"
            label_path = Path(*label_parts)
            if not label_path.is_dir():
                raise MLXUserError(
                    f"Expected labels for split '{split}' at {label_path}. "
                    "Each images/<split> directory needs a matching labels/<split> directory."
                )

        print(f"Validated dataset YAML: {data_yaml}")
        return data_yaml.parent.resolve()

In [ ]:
DATASET_ROOT = PrepareObjectDetectionDataset(
    archive_url=DATASET_ZIP_URL,
    workspace=WORKSPACE_ROOT,
).execute()
print(f"Dataset root passed to MLX: {DATASET_ROOT}")

## 5. Resolve resume behavior

MLX performs a true resume when the output project contains `last.pt` and no explicit model path is supplied. With `RESUME_LATEST=False`, this notebook refuses to reuse an experiment that already contains checkpoints; change `EXPERIMENT_NAME` to start a fresh run without deleting previous work.

In [ ]:
from mlx.modes.object_detection.artifacts import find_latest_checkpoint

latest_last_checkpoint = find_latest_checkpoint(PROJECT_DIR, pattern="last.pt")
latest_any_checkpoint = find_latest_checkpoint(PROJECT_DIR, pattern="*.pt")

if RESUME_LATEST:
    if latest_last_checkpoint is None:
        print("Resume is enabled, but no last.pt exists yet. Starting a new run.")
    else:
        print(f"MLX will resume from the latest checkpoint: {latest_last_checkpoint}")
elif latest_any_checkpoint is not None:
    raise MLXUserError(
        f"RESUME_LATEST is disabled, but {PROJECT_DIR} already contains checkpoint "
        f"{latest_any_checkpoint}. Choose a new EXPERIMENT_NAME for a fresh run, or set "
        "RESUME_LATEST=True to continue this experiment."
    )
else:
    print("Resume is disabled and the experiment has no checkpoints. Starting a new run.")

## 6. Train

This cell uses MLX's provider-neutral `TrainObjectDetectionModel` command. Keep `model_path=None`: the selected output directory is what allows MLX to discover and truly resume `last.pt`. For a resumed run, `EPOCHS` is the target total epoch count. A callback reporter writes MLX workflow events directly to notebook stdout and flushes them immediately so startup progress is visible before the training loop completes.

In [ ]:
import torch

from mlx.core.commands import CallbackWorkflowReporter
from mlx.modes.object_detection.commands import TrainObjectDetectionModel
from mlx.modes.object_detection.requests import TrainObjectDetectionRequest

if DEVICE == "auto":
    resolved_device = "cuda:0" if torch.cuda.is_available() else "cpu"
else:
    resolved_device = DEVICE

request = TrainObjectDetectionRequest(
    provider=PROVIDER,
    model=MODEL,
    model_path=None,
    dataset_path=str(DATASET_ROOT),
    output_path=str(PROJECT_DIR),
    run_name=RUN_NAME,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    height=IMAGE_HEIGHT,
    width=IMAGE_WIDTH,
    device=resolved_device,
    pretrained=PRETRAINED,
    use_best=USE_BEST,
    random_seed=RANDOM_SEED,
)

def report_training_event(event):
    print(f"[{event.level.upper()}] {event.message}", flush=True)


training_reporter = CallbackWorkflowReporter(report_training_event)
print(f"Training on: {resolved_device}", flush=True)
TRAINING_RESULTS = TrainObjectDetectionModel(
    request,
    reporter=training_reporter,
).execute()
print(
    f"Training command completed with result type: {type(TRAINING_RESULTS).__name__}",
    flush=True,
)

## 7. Locate saved checkpoints

`last.pt` contains resumable training state. `best.pt` is normally the preferred checkpoint for conversion or inference. Both remain under the selected local or Google Drive project directory.

In [ ]:
BEST_CHECKPOINT = find_latest_checkpoint(PROJECT_DIR, pattern="best.pt")
LAST_CHECKPOINT = find_latest_checkpoint(PROJECT_DIR, pattern="last.pt")

print(f"Checkpoint project: {PROJECT_DIR}")
print(f"Best checkpoint: {BEST_CHECKPOINT or 'not found'}")
print(f"Last resumable checkpoint: {LAST_CHECKPOINT or 'not found'}")